In [32]:
# CISPO stands for Clipped Importance Sampling Policy Optimization. 
# It is a cutting-edge reinforcement learning (RL) algorithm introduced by MiniMax in their MiniMax-M1
# technical report. It is heavily used in large language model (LLM) alignment, 
# especially for training long-context reasoning models (like "thinking" models similar to DeepSeek-R1).
# CISPO was specifically designed to fix a major flaw that older algorithms like PPO and GRPO suffer 
# from when dealing with complex, long-horizon reasoning

# Core difference from GRPO

#clipped_weight = torch.clamp(rho, 1 - clip_eps, 1 + clip_eps).detach()
#token_objective = clipped_weight * advantage * logp_new
#loss = -token_objective.mean()

In [33]:
import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [34]:
# 0) Load policies
# ------------------------------------------------------------

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Current trainable policy p
policy_new = copy.deepcopy(base_policy).to(device)

# For demo clarity, keep dropout off.
# Gradients still flow because requires_grad=True.
policy_new.eval()
for p in policy_new.parameters():
    p.requires_grad_(True)

# Optional frozen reference policy pi_ref
# Useful if you want KL regularization.
policy_reference = copy.deepcopy(base_policy).to(device).eval()
for p in policy_reference.parameters():
    p.requires_grad_(False)

print("Loaded:", model_name, "on", device)

Loaded: sshleifer/tiny-gpt2 on cpu


In [35]:
def encode_prompt_and_response(tokenizer, prompt_text, completion_text):
    """
    Returns:
      input_ids: [1, seq_len] for prompt+completion (teacher-forced)
      prompt_len: #tokens in prompt
      comp_len:   #tokens in completion
    """
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(prompt_text + completion_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    prompt_len = prompt_ids.shape[1]
    comp_len   = full_ids.shape[1] - prompt_len
    assert comp_len > 0, "Completion must add at least one token."
    return full_ids, prompt_len, comp_len

In [36]:
# ------------------------------------------------------------
# 2) Per-token response log-probs
# ------------------------------------------------------------

def token_logprobs_for_response(model, input_ids: torch.Tensor, prompt_len: int):
    """
    Computes per-token log-probs for response tokens.

    Causal LM convention:
        logits[:, k, :] predicts token at position k + 1

    For response token at absolute position pos:
        probability comes from logits at pos - 1.

    Returns:
        response_token_ids: [T]
        response_logps:     [T]
    """
    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    out = model(input_ids, use_cache=False)

    logits = out.logits  # [1, seq_len, vocab]
    logp_all = F.log_softmax(logits, dim=-1)

    token_ids = input_ids[0]  # [seq_len]
    seq_len = token_ids.shape[0]

    response_token_ids = token_ids[prompt_len:]  # [T]

    prev_pos = torch.arange(
        prompt_len - 1,
        seq_len - 1,
        device=model_device,
    )  # [T]

    lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

    response_logps = lp_rows.gather(
        dim=1,
        index=response_token_ids.unsqueeze(1),
    ).squeeze(1)  # [T]

    return response_token_ids, response_logps

In [37]:
def per_token_logps(model, tokenizer, prompt: str, completion: str):
    """
    Teacher-force prompt + completion and score completion tokens.
    """
    input_ids, prompt_len, _ = encode_prompt_and_response(
        tokenizer,
        prompt,
        completion,
    )

    response_ids, response_logps = token_logprobs_for_response(
        model,
        input_ids,
        prompt_len,
    )

    return response_ids, response_logps

In [38]:
# ------------------------------------------------------------
# 3) Toy reward
# ------------------------------------------------------------

def toy_reward(prompt: str, completion: str) -> float:
    """
    Toy scalar reward per completion.

    In real CISPO, this comes from a verifier or reward model.
    """
    c = completion.lower()

    reward = 0.0

    if "cat is on the" in prompt:
        if "sleep" in c:
            reward += 2.0
        if "wall" in c:
            reward += 1.0
        if "floor" in c:
            reward += 0.5
        if "runs" in c:
            reward -= 1.0
        if "mat" in c:
            reward -= 0.5

    if "2 + 2" in prompt:
        if "4" in c or "four" in c:
            reward += 2.0
        if "5" in c:
            reward -= 1.0

    # Tiny length tie-breaker
    reward += 0.05 * len(c.split())

    return float(reward)

In [39]:
# ------------------------------------------------------------
# 4) CISPO math helpers
# ------------------------------------------------------------

def group_relative_advantages(rewards: np.ndarray, eps: float = 1e-8):
    """
    Group-relative advantage:

        A_i = (r_i - mean(group_rewards)) / (std(group_rewards) + eps)

    One scalar advantage per completion.
    """
    mean_r = float(np.mean(rewards))
    std_r = float(np.std(rewards))

    advantages = (rewards - mean_r) / (std_r + eps)

    return advantages.astype(np.float64), mean_r, std_r


In [40]:
def cispo_ratio(logp_new: torch.Tensor, logp_old: torch.Tensor):
    """
    Tokenwise importance ratio:

        rho_t = exp(logp_new_t - logp_old_t)
              = pi_new(token_t | context) / pi_old(token_t | context)
    """
    return torch.exp(logp_new - logp_old)

In [41]:
def cispo_clipped_weight(
    rho: torch.Tensor,
    clip_eps: float,
):
    """
    CISPO clips the importance-sampling weight itself.

        w_t = clip(rho_t, 1-eps, 1+eps)

    Then we stop gradient through w_t.
    """
    weight = torch.clamp(
        rho,
        min=1.0 - clip_eps,
        max=1.0 + clip_eps,
    )

    return weight.detach()

In [42]:
def kl_estimator_deepseekmath(logp_new: torch.Tensor, logp_ref: torch.Tensor):
    """
    Optional positive KL estimator:

        x = logp_ref - logp_new
        ratio = exp(x) = pi_ref / pi_new
        KL_est = ratio - x - 1

    Returns:
        kl_per_token: [T]
    """
    x = logp_ref - logp_new
    ratio = torch.exp(x)

    return ratio - x - 1.0

In [43]:
# ------------------------------------------------------------
# 5) Collect rollout group
# ------------------------------------------------------------

@torch.no_grad()
def collect_cispo_rollout_group(
    policy_old,
    policy_reference,
    tokenizer,
    prompt: str,
    completions: list[str],
    rewards: np.ndarray,
):
    """
    Rollout-time CISPO quantities.

    For one prompt group:
        - detached old log-probs from policy_old
        - detached reference log-probs from policy_reference
        - group-relative advantage per completion
    """
    advantages, mean_r, std_r = group_relative_advantages(rewards)

    rollout_items = []

    for i, completion in enumerate(completions):
        ids_old, logp_old = per_token_logps(
            policy_old,
            tokenizer,
            prompt,
            completion,
        )

        ids_ref, logp_ref = per_token_logps(
            policy_reference,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_old, ids_ref), (
            "Token mismatch between old policy and reference scoring."
        )

        rollout_items.append(
            {
                "completion": completion,
                "token_ids": ids_old.detach().cpu(),
                "logp_old": logp_old.detach().cpu(),
                "logp_ref": logp_ref.detach().cpu(),
                "reward": float(rewards[i]),
                "advantage": float(advantages[i]),
            }
        )

    group_stats = {
        "mean_reward": mean_r,
        "std_reward": std_r,
        "advantages": advantages,
    }

    return rollout_items, group_stats

In [44]:
# ------------------------------------------------------------
# 6) CISPO objective computation
# ------------------------------------------------------------

def cispo_compute_objective(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    clip_eps: float,
    beta_kl: float = 0.0,
):
    """
    Computes CISPO objective over one prompt group.

    For each completion i:
        A_i = group-relative scalar advantage
        A_i is broadcast to all tokens in that completion

        rho_t = exp(logp_new_t - logp_old_t)

        clipped_weight_t = clip(rho_t, 1-eps, 1+eps).detach()

        token_objective_t =
            clipped_weight_t * A_i * logp_new_t

        optional:
            token_objective_t -= beta_kl * KL_t

    Completion objective:
        mean over tokens

    Group objective:
        mean over completions

    Loss:
        loss = -objective
    """
    per_completion_objectives = []

    all_rhos = []
    all_weights = []
    all_kls = []

    for item in rollout_items:
        completion = item["completion"]

        token_ids_old = item["token_ids"].to(device)
        logp_old = item["logp_old"].to(device)
        logp_ref = item["logp_ref"].to(device)

        adv_scalar = item["advantage"]

        ids_new, logp_new = per_token_logps(
            policy_new,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(ids_new, token_ids_old), (
            "Token mismatch between rollout tokens and current-policy scoring."
        )

        T = logp_new.shape[0]

        adv_tokens = torch.full(
            size=(T,),
            fill_value=adv_scalar,
            device=device,
            dtype=logp_new.dtype,
        )

        rho = cispo_ratio(
            logp_new,
            logp_old,
        )

        clipped_weight = cispo_clipped_weight(
            rho,
            clip_eps=clip_eps,
        )

        token_objective = clipped_weight * adv_tokens * logp_new

        if beta_kl > 0.0:
            kl = kl_estimator_deepseekmath(
                logp_new,
                logp_ref,
            )

            token_objective = token_objective - beta_kl * kl
            all_kls.append(kl.detach())

        completion_objective = token_objective.mean()

        per_completion_objectives.append(completion_objective)

        all_rhos.append(rho.detach())
        all_weights.append(clipped_weight.detach())

    objective = torch.stack(per_completion_objectives).mean()
    loss = -objective

    rho_cat = torch.cat(all_rhos)
    weight_cat = torch.cat(all_weights)

    with torch.no_grad():
        clipfrac = (
            ((rho_cat > 1.0 + clip_eps) | (rho_cat < 1.0 - clip_eps))
            .float()
            .mean()
            .item()
        )

        if len(all_kls) > 0:
            kl_cat = torch.cat(all_kls)
            mean_kl = float(kl_cat.mean().cpu())
        else:
            mean_kl = 0.0

        stats = {
            "loss": float(loss.detach().cpu()),
            "objective": float(objective.detach().cpu()),
            "mean_rho": float(rho_cat.mean().cpu()),
            "min_rho": float(rho_cat.min().cpu()),
            "max_rho": float(rho_cat.max().cpu()),
            "mean_clipped_weight": float(weight_cat.mean().cpu()),
            "clipfrac": clipfrac,
            "mean_kl": mean_kl,
        }

    return loss, stats


In [45]:
# ------------------------------------------------------------
# 7) CISPO update step
# ------------------------------------------------------------

def cispo_update_step(
    policy_new,
    optimizer,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    clip_eps: float,
    beta_kl: float = 0.0,
):
    """
    One CISPO optimization step over one prompt group.
    """
    loss, stats = cispo_compute_objective(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        clip_eps=clip_eps,
        beta_kl=beta_kl,
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return stats

In [46]:
# ------------------------------------------------------------
# 8) Evaluation helper
# ------------------------------------------------------------

@torch.no_grad()
def cispo_eval_stats(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    clip_eps: float,
    beta_kl: float = 0.0,
):
    """
    Compute CISPO stats without updating.
    """
    loss, stats = cispo_compute_objective(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        clip_eps=clip_eps,
        beta_kl=beta_kl,
    )

    return stats


In [47]:
# ------------------------------------------------------------
# 9) CISPO settings
# ------------------------------------------------------------

num_cispo_iterations = 2

clip_eps = 0.2

# Optional KL regularization.
# Set to 0.0 for pure CISPO mechanics.
beta_kl = 0.0

lr = 1e-5

optimizer = torch.optim.Adam(
    policy_new.parameters(),
    lr=lr,
)


# ------------------------------------------------------------
# 10) Two prompt groups, batch size = 1 prompt group
# ------------------------------------------------------------

examples = [
    {
        "prompt": "cat is on the",
        "completions": [
            " wall and its sleeping.",
            " mat and it runs.",
            " floor.",
        ],
    },
    {
        "prompt": "2 + 2 =",
        "completions": [
            " 4.",
            " 5.",
            " four.",
        ],
    },
]


# ------------------------------------------------------------
# 11) Main CISPO flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"CISPO EXAMPLE {example_idx} / BATCH SIZE = 1 PROMPT GROUP")
    print("=" * 80)

    prompt = ex["prompt"]
    completions = ex["completions"]

    print("Prompt:", repr(prompt))
    print("Completions:")
    for completion in completions:
        print("   ", repr(completion))

    # --------------------------------------------------------
    # A) Create rollout snapshot q for this prompt group
    # --------------------------------------------------------
    policy_old = copy.deepcopy(policy_new).to(device).eval()
    for p in policy_old.parameters():
        p.requires_grad_(False)

    print("\n[1] Rollout snapshot")
    print("    q = policy_old = frozen copy of current policy_new")
    print("    q is fixed for this prompt group.")

    # --------------------------------------------------------
    # B) Compute rewards for the completion group
    # --------------------------------------------------------
    print("\n[2] Compute group rewards")

    rewards = np.array(
        [toy_reward(prompt, completion) for completion in completions],
        dtype=np.float64,
    )

    print("    rewards:", rewards)

    # --------------------------------------------------------
    # C) Collect rollout log-probs and group-relative advantages
    # --------------------------------------------------------
    print("\n[3] Collect rollout log-probs and group-relative advantages")

    rollout_items, group_stats = collect_cispo_rollout_group(
        policy_old,
        policy_reference,
        tokenizer,
        prompt,
        completions,
        rewards,
    )

    print("    mean_reward:", group_stats["mean_reward"])
    print("    std_reward: ", group_stats["std_reward"])
    print("    advantages: ", group_stats["advantages"])

    for item in rollout_items:
        print(
            "    completion:",
            repr(item["completion"]),
            "reward:",
            item["reward"],
            "advantage:",
            item["advantage"],
        )

    # --------------------------------------------------------
    # D) Before first CISPO update
    # --------------------------------------------------------
    print("\n[4] Before CISPO update")

    before = cispo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        clip_eps=clip_eps,
        beta_kl=beta_kl,
    )

    print("    rho mean/min/max:",
          before["mean_rho"], before["min_rho"], before["max_rho"])
    print("    mean clipped weight:",
          before["mean_clipped_weight"])
    print("    objective:", before["objective"])
    print("    loss:     ", before["loss"])
    print("    mean_kl:  ", before["mean_kl"])
    print("    clipfrac: ", before["clipfrac"])

    # --------------------------------------------------------
    # E) CISPO update iterations on same rollout group
    # --------------------------------------------------------
    print("\n[5] CISPO update loop")
    print("    num_cispo_iterations:", num_cispo_iterations)
    print("    logp_old/q stays fixed during these iterations.")
    print("    logp_new/p is recomputed each iteration.")
    print("    clip_eps:", clip_eps)

    for cispo_iter in range(1, num_cispo_iterations + 1):

        stats = cispo_update_step(
            policy_new,
            optimizer,
            tokenizer,
            prompt,
            rollout_items,
            clip_eps=clip_eps,
            beta_kl=beta_kl,
        )

        print(f"\n    CISPO iteration {cispo_iter}")
        print("        rho mean/min/max:",
              stats["mean_rho"], stats["min_rho"], stats["max_rho"])
        print("        mean clipped weight:",
              stats["mean_clipped_weight"])
        print("        objective:", stats["objective"])
        print("        loss:     ", stats["loss"])
        print("        mean_kl:  ", stats["mean_kl"])
        print("        clipfrac: ", stats["clipfrac"])

    # --------------------------------------------------------
    # F) After update, evaluate same rollout group
    # --------------------------------------------------------
    print("\n[6] After CISPO update on same rollout group")

    after = cispo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        clip_eps=clip_eps,
        beta_kl=beta_kl,
    )

    print("    rho mean/min/max:",
          after["mean_rho"], after["min_rho"], after["max_rho"])
    print("    mean clipped weight:",
          after["mean_clipped_weight"])
    print("    objective:", after["objective"])
    print("    loss:     ", after["loss"])
    print("    mean_kl:  ", after["mean_kl"])
    print("    clipfrac: ", after["clipfrac"])

    print("\n[7] Discard this rollout group")
    print("    Next prompt group will create a fresh policy_old snapshot.")


CISPO EXAMPLE 1 / BATCH SIZE = 1 PROMPT GROUP
Prompt: 'cat is on the'
Completions:
    ' wall and its sleeping.'
    ' mat and it runs.'
    ' floor.'

[1] Rollout snapshot
    q = policy_old = frozen copy of current policy_new
    q is fixed for this prompt group.

[2] Compute group rewards
    rewards: [ 3.2  -1.3   0.55]

[3] Collect rollout log-probs and group-relative advantages
    mean_reward: 0.8166666666666668
    std_reward:  1.84676895023594
    advantages:  [ 1.29054223 -1.1461459  -0.14439633]
    completion: ' wall and its sleeping.' reward: 3.2 advantage: 1.290542230593286
    completion: ' mat and it runs.' reward: -1.3 advantage: -1.1461458971003309
    completion: ' floor.' reward: 0.55 advantage: -0.1443963334929551

[4] Before CISPO update
    rho mean/min/max: 1.0 1.0 1.0
    mean clipped weight: 1.0
    objective: 0.009905815124511719
    loss:      -0.009905815124511719
    mean_kl:   0.0
    clipfrac:  0.0

[5] CISPO update loop
    num_cispo_iterations: 2
    